# IdiomX – Prepare dataset for Task 2: Context-to-Idiom Retrieval

This notebook presents prparing the clean dataset for **Task 2** benchmark.

---

## Task definition

Given an **English idiomatic contextual sentence**, the goal is to retrieve the correct **canonical idiom** from a fixed idiom inventory.

This is a **closed-set retrieval task**:
- **Input**: idiomatic contextual sentence  
- **Output**: canonical idiom  
- **Prediction space**: fixed idiom bank  

---

In [1]:
# [1.1] Environment setup and reproducibility

from pathlib import Path
import warnings
import random
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

print("Environment setup complete.")
print(f"Random seed: {SEED}")

Environment setup complete.
Random seed: 42


In [87]:
# [2.1] Load IdiomX high-quality dataset (minimal output)

from datasets import load_dataset

HF_DATASET_ID = "aymansharara/IdiomX"
CONFIG_NAME = "idiomx_full"

dataset = load_dataset(HF_DATASET_ID, CONFIG_NAME)

print("Dataset loaded successfully.")
print("Available splits:", list(dataset.keys()))

Dataset loaded successfully.
Available splits: ['full']


In [88]:
# [2.2] Inspect dataset structure (light preview)

df = dataset[list(dataset.keys())[0]].to_pandas()

print("Shape:", df.shape)
print("Columns:", list(df.columns))

df.head(2)

Shape: (196343, 63)
Columns: ['idiom_id', 'idiom_canonical', 'idiom_surface', 'example', 'example_usage_label', 'is_example_idiom', 'idiom_canonical_meaning', 'idiom_canonical_meaning_arabic', 'idiom_canonical_meaning_french', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_arabic', 'idiom_in_example_meaning_french', 'example_raw', 'example_normalized', 'example_language', 'meaning_language', 'source', 'source_type', 'source_url', 'record_origin', 'license_source', 'source_dataset', 'pos', 'tags', 'idiom_confidence', 'is_idiom', 'idiom_validity_label', 'ambiguity_flag', 'idiom_compositionality_level', 'idiom_register', 'idiom_domain', 'learner_difficulty', 'slang_strength', 'regionality', 'offensive_flag', 'idiom_in_example_arabic', 'idiom_in_example_french', 'idiom_level_explanation_en', 'idiom_level_explanation_ar', 'idiom_level_explanation_fr', 'explanation_en', 'explanation_ar', 'explanation_fr', 'hard_negative_idioms', 'meaning_paraphrases_en', 'meaning_paraphrases_ar', '

,idiom_id,idiom_canonical,idiom_surface,example,example_usage_label,is_example_idiom,idiom_canonical_meaning,idiom_canonical_meaning_arabic,idiom_canonical_meaning_french,idiom_in_example_meaning_en,idiom_in_example_meaning_arabic,idiom_in_example_meaning_french,example_raw,example_normalized,example_language,meaning_language,source,source_type,source_url,record_origin,license_source,source_dataset,pos,tags,idiom_confidence,is_idiom,idiom_validity_label,ambiguity_flag,idiom_compositionality_level,idiom_register,idiom_domain,learner_difficulty,slang_strength,regionality,offensive_flag,idiom_in_example_arabic,idiom_in_example_french,idiom_level_explanation_en,idiom_level_explanation_ar,idiom_level_explanation_fr,explanation_en,explanation_ar,explanation_fr,hard_negative_idioms,meaning_paraphrases_en,meaning_paraphrases_ar,meaning_paraphrases_fr,is_generated_example,enrichment_model,enrichment_version,validation_status,context_type,source_style,minimal_pair_id,paraphrase_group_id,is_adversarial_example,adversarial_type,expected_label,row_type,sentence_length_chars,sentence_length_words,semantic_similarity_example_vs_meaning,semantic_quality
0,idiomx_543ff4605a0e,'ark at 'ee,'ark at 'ee,"’Ark at ’ee, as if you could ever finish on time!",idiomatic,True,"An exclamation used to draw attention to what someone is saying or to express surprise at their words; essentially, it means 'listen to you!' or 'listen to that!'","تعجب يُقال لجذب الانتباه إلى ما يقوله شخص ما أو للتعبير عن الدهشة من كلامه، بمعنى ""استمع إلى ما تقول!"" أو ""انظر إلى ذلك!""",None,"Sarcastic remark casting doubt on someone's ability, emphasizing surprise.",تعليق ساخر يشكك في قدرة شخص ما، ويؤكد الدهشة.,None,"‘Look at that water! No wonder Duddle said he wouldn’t dare take the raft down this way; it’s dreadful!’ / Togget pointed ahead. ‘Yurr oi think et wursens yonder, ’ark at ee roaren et makes!’",ark at ee as if you could ever finish on time,en,en,kaikki_wiktionary,dictionary,None,llm_enriched_v2,wiktionary_cc_by_sa_4_0,idiomx_main,phrase,informal,llm_enriched,True,valid,strongly_idiomatic,opaque,informal,regional,hard,none,general,False,انظر إلى ذلك، كما لو كنت ستنتهي في الوقت المحدد!,None,This phrase is idiomatic because it uses dialectal contraction and an exclamatory phrase that does not literally mean to physically 'look at the ear' but instead commands attention to someone's sp...,"هذه العبارة تعبير مجازي لأنها تستخدم اختصارًا لهجة وعبارة تعجبية لا تعني حرفيًا ""انظر إلى الأذن"" بل تأمر بالانتباه إلى كلام أو رأي شخص ما.",None,The expression is used sarcastically to mock or express disbelief in a claim.,يُستخدم التعبير للسخرية أو التعبير عن عدم التصديق في ادعاء.,None,"[""listen up"", ""hear me out"", ""look who's talking""]","[""Pay attention to what is being said"", ""Listen to that surprising statement"", ""Consider the speaker's words""]","[""انتبه إلى ما يقال"", ""استمع إلى ذلك القول المفاجئ"", ""اعتبر كلمات المتحدث""]",None,True,gpt-4.1-mini-2025-04-14,v2,valid,sarcastic,synthetic_narrative,pair_86768bd5179a,paraphrase_6a8ebd5786f7,False,None,idiomatic,main_example,49,11,-0.006854,low
1,idiomx_543ff4605a0e,'ark at 'ee,'ark at 'ee,’Ark at ’ee! What do you mean by quitting so suddenly?,idiomatic,True,"An exclamation used to draw attention to what someone is saying or to express surprise at their words; essentially, it means 'listen to you!' or 'listen to that!'","تعجب يُقال لجذب الانتباه إلى ما يقوله شخص ما أو للتعبير عن الدهشة من كلامه، بمعنى ""استمع إلى ما تقول!"" أو ""انظر إلى ذلك!""",None,Exclamation to express surprise when asking a question.,تعجب للتعبير عن الدهشة أثناء طرح سؤال.,None,"‘Look at that water! No wonder Duddle said he wouldn’t dare take the raft down this way; it’s dreadful!’ / Togget pointed ahead. ‘Yurr oi think et wursens yonder, ’ark at ee roaren et makes!’",ark at ee what do you mean by quitting so suddenly,en,en,kaikki_wiktionary,dictionary,None,llm_enriched_v2,wiktionary_cc_by_sa_4_0,idiomx_main,phrase,informal,llm_enriched,True,vali

In [94]:
# [2.3] Inspect core Task 2 quality fields

print("example_usage_label:")
print(df["example_usage_label"].value_counts(dropna=False))

print("\nis_example_idiom:")
print(df["is_example_idiom"].value_counts(dropna=False))

print("\nvalidation_status:")
print(df["validation_status"].value_counts(dropna=False))

print("\nsemantic_quality:")
print(df["semantic_quality"].value_counts(dropna=False).head(20))

example_usage_label:
example_usage_label
literal       90041
idiomatic     89722
borderline    16580
Name: count, dtype: int64

is_example_idiom:
is_example_idiom
False    106609
True      89734
Name: count, dtype: int64

validation_status:
validation_status
valid          180260
final_ready     10311
verified         3462
corrected        2310
Name: count, dtype: int64

semantic_quality:
semantic_quality
high                  130656
medium                 43001
low                    12375
synthetic_unscored     10311
Name: count, dtype: int64


In [95]:
# [2.4] Inspect what we would REMOVE

removed_df = df[
    (df["semantic_quality"] != "high") |
    (df["example_usage_label"] == "borderline")
]

print("Removed size:", len(removed_df))

# sample
removed_sample = removed_df.sample(20, random_state=42)[
    ["idiom_canonical", "example", "example_usage_label", "semantic_quality", "validation_status"]
]

removed_sample

Removed size: 73332


,idiom_canonical,example,example_usage_label,semantic_quality,validation_status
41021,crawl up someone's ass,"He was crawling up my ass with compliments, and I wasn’t sure if it was genuine.",borderline,medium,valid
4974,I'm lost,"While assembling the furniture, I realized I’m lost without the instruction manual.",literal,low,valid
5483,Irishman's hurricane,"Yeah, right—another Irishman's hurricane at the office when the boss finally left! Dead silence... so unusual!",idiomatic,medium,valid
153996,sweet smell of success,"Despite the setbacks, there was no sweet smell of success this time.",borderline,medium,valid
54931,fear not,"As the sun set, the hero said, “Fear not, for dawn will break soon.”",idiomatic,medium,valid
54623,farm upstate,"Oh sure, just send all your problems to the farm upstate—they solve themselves!",idiomatic,medium,valid
177206,weak sister,"He certainly wasn’t the weak sister this time, despite past failures.",idiomatic,medium,valid
95481,lick the pants off,"He almost licked the pants off his opponent, but the game ended in a draw.",borderline,high,valid
95704,life support,"The CEO claims the merger put the company on life support, but profits increased anyway.",borderline,low,valid
193215,vibe shift moment,"Nice vibe shift moment there, the power outage really brightened up the office!",literal,synthetic_unscored,final_ready


In [96]:
# [2.5] Check high-confidence idioms outside "high"

suspect_good = df[
    (df["semantic_quality"] != "high") &
    (df["example_usage_label"] == "idiomatic") &
    (df["validation_status"].isin(["verified", "corrected"]))
]

print("Potential GOOD but not high-quality:", len(suspect_good))

suspect_good.sample(10)[
    ["idiom_canonical", "example", "semantic_quality", "validation_status"]
]

Potential GOOD but not high-quality: 1296


,idiom_canonical,example,semantic_quality,validation_status
89363,keep one's chin up,"Keep your chin up... yeah, right.",medium,corrected
55247,feel one's way,"When starting the new project, we'll have to feel our way through uncharted territory.",medium,corrected
102291,make up one's mind,"I told him to make up his mind already, enough hesitation!",medium,corrected
145914,smoke someone's pole,"LOL, these guys promise to smoke your pole all night at the party!",low,verified
162313,the pot calling the kettle black,pot calling the kettle black,medium,corrected
138176,scratch one's own itch,Everyone should scratch their own itch and build what they really need! #innovation #DIY,medium,corrected
73272,haul someone over the coals,The manager hauled the employee over the coals for missing the project deadline.,low,corrected
91264,kiss someone's ass,She always tries to kiss her ass off to the boss to get a promotion.,medium,corrected
127112,pull one's finger out,You need to pull your finger out and finish the project before the deadline.,medium,corrected
11017,adorn oneself with borrowed plumes,"Oh great, he’s adorning himself with borrowed plumes again as if he did all the work!",low,verified


### **we have good idioms labeled as medium/low**

Examples:

* _“make up one's mind”_ → medium + corrected - GOOD
* _“pull one's finger out”_ → medium + corrected - GOOD
* _“haul someone over the coals”_ → low + corrected - still GOOD
* _“pot calling the kettle black”_ → medium + corrected - GOOD

-So:
-filtering only `semantic_quality == high` is TOO AGGRESSIVE
we would lose real idioms

### But you ALSO have real noise

From removed sample:

* borderline junk:* _“lick the pants off”_ → borderline 
* _“lick the pants off”_ → borderline 
* weak literal:* _“I'm lost”_ → low 
* _“I'm lost”_ → low 
* synthetic:* _“vibe shift moment”_ → synthetic\_unscored 
* _“vibe shift moment”_ → synthetic\_unscored 

So filtering is still necessary

In [97]:
# [2.6] FINAL CLEAN DATASET (balanced + safe)

clean_df = df[
    (
        # keep strong quality
        (df["semantic_quality"] == "high") |

        # OR trusted human/validated data
        (df["validation_status"].isin(["verified", "corrected"]))
    )
    &
    # remove ambiguous usage
    (df["example_usage_label"] != "borderline")
].copy()

print("Original size:", len(df))
print("Clean size:", len(clean_df))
print("Kept %:", round(len(clean_df) / len(df) * 100, 2), "%")

# sanity checks
print("\nsemantic_quality:")
print(clean_df["semantic_quality"].value_counts())

print("\nvalidation_status:")
print(clean_df["validation_status"].value_counts())

print("\nexample_usage_label:")
print(clean_df["example_usage_label"].value_counts())

Original size: 196343
Clean size: 124674
Kept %: 63.5 %

semantic_quality:
semantic_quality
high      123011
medium      1266
low          397
Name: count, dtype: int64

validation_status:
validation_status
valid        118902
verified       3462
corrected      2310
Name: count, dtype: int64

example_usage_label:
example_usage_label
literal      70205
idiomatic    54469
Name: count, dtype: int64


In [98]:
# [2.7] Inspect REMOVED data (what we dropped)

removed_df = df.drop(clean_df.index)

print("Removed size:", len(removed_df))

removed_sample = removed_df.sample(30, random_state=42)[
    [
        "idiom_canonical",
        "example",
        "example_usage_label",
        "semantic_quality",
        "validation_status"
    ]
]

removed_sample

Removed size: 71669


,idiom_canonical,example,example_usage_label,semantic_quality,validation_status
172553,two left hands,"He literally has two left hands growing from his arms, making him awkward.",borderline,medium,valid
186755,on the fast track,"Yeah, the train is literally on the fast track — if by fast you mean barely moving at all.",literal,synthetic_unscored,final_ready
30615,buy a ticket to,Just bought a ticket to the game next Friday! Can't wait!,literal,medium,valid
101237,make it rain,It's time to make it rain at the club tonight!,idiomatic,low,valid
5282,Ich dien,Does 'Ich dien' literally mean 'I serve' in German?,literal,medium,valid
51342,ejusdem generis,"Oh sure, just apply some Latin magic like 'ejusdem generis' and everything becomes crystal clear!",idiomatic,medium,valid
107759,nigger rich,"Oh sure, you're totally nigger rich now, eating caviar on a ramen budget.",idiomatic,medium,valid
132259,red flag,Her constant canceling of plans was a total red flag to me.,idiomatic,medium,valid
3944,I tell you,"I tell you, do you think this is the best way to handle it?",idiomatic,low,valid
45800,do not want,I do not want any more coffee; I’ve had six cups already.,literal,medium,valid


In [100]:
# [2.8] Count potentially valuable idiomatic rows removed

removed_df = df.drop(clean_df.index)

removed_good = removed_df[
    (removed_df["example_usage_label"] == "idiomatic") &
    (removed_df["validation_status"].isin(["verified", "corrected"]))
].copy()

print("Potentially valuable removed rows:", len(removed_good))

removed_good_sample = removed_good.sample(min(20, len(removed_good)), random_state=42)[
    ["idiom_canonical", "example", "semantic_quality", "validation_status"]
]

removed_good_sample

Potentially valuable removed rows: 0


,idiom_canonical,example,semantic_quality,validation_status


In [102]:
clean_df.shape

(124674, 65)

In [103]:
# [2.9] Save Task 2 cleaned dataset to root/data/

from pathlib import Path

# Go up one level from notebooks/ → project root, then into data/
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

csv_path = DATA_DIR / "task2_retrieval_clean_v1.csv"
parquet_path = DATA_DIR / "task2_retrieval_clean_v1.parquet"

clean_df.to_csv(csv_path, index=False)
clean_df.to_parquet(parquet_path, index=False)

print("Saved:")
print(f"- {csv_path.resolve()}")
print(f"- {parquet_path.resolve()}")
print("Rows:", len(clean_df))
print("Unique idioms:", clean_df["idiom_canonical"].nunique())

Saved:
- task2_retrieval_clean_v1.csv
- task2_retrieval_clean_v1.parquet
Rows: 124674
Unique idioms: 13803
